# net 模型：损失、优化器、TabM 与参数搜索

安装当前源码：`pip install -e ".[net,tune]"`。本例使用真实放款数据的完整 970 行（以读取结果为准），分为训练、验证和独立测试集。

可训练模型为 ResNet、FTTransformer、TabM；TabPFN 在当前封装中采用预训练上下文推理，搜索其原生推理参数。本例采用少量轮数和试验验证调用链，不作为模型效果基准。

## 1. 数据与环境

In [ ]:
from pathlib import Path
from importlib.metadata import version
import sys
import numpy as np
import pandas as pd
import torch
from torch.nn import functional as F
from sklearn.model_selection import train_test_split
from IPython.display import display
from hscredit import ResNet, FTTransformer, TabM, TabPFN, FocalLoss, ModelTuner, AutoTuner, Real

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "examples/hscredit_yyp.xlsx").exists())
data = pd.read_excel(root / "examples/hscredit_yyp.xlsx")
features = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24", "商品类别"]
training, test = train_test_split(data, test_size=0.2, stratify=data.FPD, random_state=42)
train, valid = train_test_split(training, test_size=0.2, stratify=training.FPD, random_state=42)
assert len(train) + len(valid) + len(test) == len(data)
display(pd.DataFrame({"数据集": ["训练", "验证", "独立测试"], "样本数": [len(train), len(valid), len(test)]}))
display(pd.DataFrame({"组件": ["Python", "torch", "tabm", "optuna", "tabpfn"],
                      "版本": [sys.version.split()[0], version("torch"), version("tabm"), version("optuna"), version("tabpfn")]}))

## 2. 自定义 PyTorch 损失与优化器

损失接收 `(raw_prediction, target)` 两个一维张量，二分类输入是原始 logits；必须返回逐样本损失，不能断开计算图。统一训练层负责额外样本权重和均值。优化器通过名称或 `torch.optim` 类选择，特有参数放在 `optimizer_params`。

In [ ]:
def weighted_bce(logits, target, positive_weight=2.0):
    values = F.binary_cross_entropy_with_logits(logits, target, reduction="none")
    return values * torch.where(target == 1, positive_weight, 1.0)

resnet = ResNet(
    d_block=32, n_blocks=2, max_epochs=10, early_stopping_rounds=3,
    loss=weighted_bce, loss_params={"positive_weight": 2.0},
    optimizer="sgd", optimizer_params={"momentum": 0.9, "nesterov": True},
    learning_rate=0.01, weight_decay=0.0001, random_state=42,
)
resnet.fit(train[features], train.FPD, sample_weight=np.ones(len(train)), eval_set={"验证": (valid[features], valid.FPD)})
assert resnet.optimizer_.defaults["momentum"] == 0.9
display(pd.DataFrame([resnet.evaluate(test[features], test.FPD, metrics=["auc", "ks", "logloss"])]))
display(resnet.get_learning_curve().tail())

## 3. TabM 的成员表示与 HSCredit 损失

TabM 按成员分别计算损失再取平均，分类推理对成员概率取平均。`get_embeddings` 默认返回平均表示；`aggregate=None` 保留成员轴。HSCredit BaseLoss 通过解析梯度适配；要求 `is_additive=True`，不直接接受依赖总体排序或绑定样本数组的损失。

In [ ]:
tabm = TabM(
    d_block=32, n_blocks=2, k=8, max_epochs=10, early_stopping_rounds=3,
    num_embeddings="linear_relu", num_embedding_params={"d_embedding": 8},
    loss=FocalLoss(alpha=0.75, gamma=2.0), optimizer="adamw",
    optimizer_params={"betas": (0.9, 0.99)}, learning_rate=0.002, weight_decay=0.0003,
    random_state=42,
)
tabm.fit(train[features], train.FPD, eval_set={"验证": (valid[features], valid.FPD)})
embedding = tabm.get_embeddings(test[features])
member_embeddings = tabm.get_embeddings(test[features], aggregate=None)
assert member_embeddings.shape == (8, len(test), 32)
np.testing.assert_allclose(embedding, member_embeddings.mean(axis=0))
probability = tabm.predict_proba(test[features])
assert probability.shape == (len(test), 2)
np.testing.assert_allclose(probability.sum(axis=1), 1)
display(pd.DataFrame([{"样本数": len(test), "成员数": tabm.k, "表示维度": embedding.shape[1],
                       **tabm.evaluate(test[features], test.FPD, metrics=["auc", "ks", "logloss"])}]))

## 4. 条件搜索：优化器选择与专用参数

训练损失 `loss`、搜索指标 `metric`、训练后评估表 `eval_metric` 相互独立。不同优化器支持不同参数，使用搜索函数返回适用配置；不能把 SGD 的 momentum 传给 AdamW。TabM 的 k 默认固定，分别比较不同 k 的搜索更易解释。外部验证用于早停和选优，测试集只在最终评价时使用。

In [ ]:
def classifier_space(trial):
    optimizer = trial.suggest_categorical("optimizer", ["adamw", "sgd"])
    config = {
        "optimizer": optimizer,
        "optimizer_params": {},
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True),
        "d_block": trial.suggest_categorical("d_block", [32, 64]),
        "loss_params__gamma": trial.suggest_categorical("loss_params__gamma", [1.0, 2.0]),
    }
    if optimizer == "sgd":
        config["optimizer_params__momentum"] = trial.suggest_float("optimizer_params__momentum", 0.5, 0.95)
    else:
        config["optimizer_params"] = {"betas": (0.9, 0.99)}
    return config

best_classifier = TabM(k=8, n_blocks=2, max_epochs=8, early_stopping_rounds=3,
                      loss="focal", loss_params={"alpha": 0.75}, random_state=42).tune(
    train[features], train.FPD, eval_set={"验证": (valid[features], valid.FPD)},
    search_space=classifier_space, metric="auc", n_trials=3, n_jobs=1, show_progress_bar=False,
    trial_points=[{"optimizer": "adamw"}, {"optimizer": "sgd"}],
)
assert best_classifier.validation_fraction == 0
assert best_classifier.early_stopping_rounds is None
assert best_classifier.training_summary_["样本数"] == len(train)
classification_history = best_classifier.tuner.get_optimization_history()
classification_columns = {"number": "试验编号", "value_AUC": "验证AUC", "params_optimizer": "优化器",
                          "params_learning_rate": "学习率", "params_d_block": "隐藏层宽度",
                          "params_loss_params__gamma": "聚焦系数", "params_optimizer_params__momentum": "动量"}
display(classification_history.reindex(columns=classification_columns).rename(columns=classification_columns))
display(pd.DataFrame([best_classifier.evaluate(test[features], test.FPD, metrics=["auc", "ks", "logloss"])]))

## 5. 回归调参与嵌套参数

回归调参按连续预测计算 RMSE/MAE/R²，不调用分类概率接口或计算分类 LIFT。内部训练目标会标准化，Huber 的 delta 在标准化空间解释；公开预测和最终指标使用原始金额单位。多折早停只使用各训练折内部划分，外层验证折用于选优。

In [ ]:
regressor = FTTransformer(
    objective="regression", d_block=32, n_blocks=1, max_epochs=8, early_stopping_rounds=3,
    loss="huber", loss_params={"delta": 1.0}, optimizer="adam", random_state=42,
)
best_regressor = regressor.tune(
    training[features], training["放款金额"], cv=2, metric="rmse", n_trials=2, n_jobs=1,
    search_space={"learning_rate": Real(1e-4, 3e-3, prior="log-uniform"), "loss_params__delta": [0.5, 1.0, 2.0]},
    show_progress_bar=False,
)
prediction = best_regressor.predict(test[features])
metrics = best_regressor.evaluate(test[features], test["放款金额"])
np.testing.assert_allclose(metrics["均方根误差"], np.sqrt(np.mean((test["放款金额"].to_numpy()-prediction)**2)))
oof = best_regressor.tuner.get_oof_predictions()
assert "预测值" in oof.columns and "预测概率" not in oof.columns
assert (oof["验证次数"] == 1).all()
display(pd.DataFrame([metrics]))
display(oof.head())
regression_history = best_regressor.tuner.get_optimization_history()
regression_columns = {"number": "试验编号", "value_均方根误差": "验证均方根误差",
                      "params_learning_rate": "学习率", "params_loss_params__delta": "损失阈值"}
display(regression_history.reindex(columns=regression_columns).rename(columns=regression_columns))

## 6. TabPFN 的调参边界

当前封装不进行 TabPFN 梯度微调，因此不接受 loss、optimizer、max_epochs。可以用同一调参接口搜索 n_estimators、softmax_temperature 等原生推理参数。以下使用官方真实权重；首次运行可能下载所选权重。

In [ ]:
pfn_tuner = AutoTuner.create(
    "tabpfn", metric="auc", n_jobs=1, random_state=42,
    model_params={"device": "auto", "inference_precision": torch.float32},
    search_space={"n_estimators": [1], "softmax_temperature": [0.8, 1.0]},
)
pfn_tuner.fit(train[features], train.FPD, eval_set={"验证": (valid[features], valid.FPD)}, n_trials=1, show_progress_bar=False)
best_pfn = pfn_tuner.get_best_model()
assert best_pfn.get_embeddings(test[features].iloc[:5]).shape[0] == 5
display(pd.DataFrame([best_pfn.evaluate(test[features], test.FPD, metrics=["auc", "ks", "logloss"])]))

## 7. 保存搜索后的模型

In [ ]:
output_dir = root / ".audit_tmp" / "tabm-net"
output_dir.mkdir(parents=True, exist_ok=True)
for name, model in [("自定义损失", resnet), ("TabM分类", best_classifier), ("FTTransformer回归", best_regressor)]:
    path = output_dir / f"{name}.pkl"
    model.save(path)
    restored = type(model).load(path)
    np.testing.assert_allclose(restored.predict(test[features].iloc[:5]), model.predict(test[features].iloc[:5]))
    np.testing.assert_allclose(restored.transform(test[features].iloc[:5]), model.transform(test[features].iloc[:5]))
print("自定义损失、优化器、分类/回归搜索和制品恢复验证通过")

参考：[TabM 官方实现及训练协议](https://github.com/yandex-research/tabm)、[PyTorch 优化器](https://docs.pytorch.org/docs/stable/optim.html)。